- 4a Energy slicing
- 4b Slice fitting
- 4c Neutron channel bounds
- 4d Count rate and dwell time

### Imports/Constants

In [ ]:
# Importing needed code

import re
from typing import (
    Callable,
    TypeVar,
    Any,
    Literal,
    TypeGuard
)
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib as mpl
import mpl_toolkits.mplot3d.art3d as art3d
import pandas as pd
import numpy as np

from data_processing.paths import get_exp_root
from data_processing.dataframe_validation import (
    DetectorDataframeColumn,
    BinningDataframeColumn,
    EnergyColumn,
    get_df_col
)
from data_processing.experiment_data_keys import (
    ExperimentDataKey,
    ExperimentNeutronData
)
from data_processing.loading.dataframe_loading import load_parquet_psd
from data_processing.loading.timetag_processing import (
    calculate_timetag_hours,
    calculate_event_time
)
from data_processing import processing as proc
from data_processing import types as proc_types
from data_processing import helpers
from data_processing.processing.neutron_window_strategy.strategy_factory import \
    NeutronStrategyFactory
from data_processing.processing.neutron_window_strategy.abstract_strategy import \
    AbstractNeutronStrategy
from data_processing.processing.calibration import DetectorCalibrationParams
from data_processing.types import (
    NasaGenerationSettings,
    NeutronDistributionGenerationSettings,
    NeutronWindowSettings,
    WindowType,
)
from data_processing.loading.window_loading import (
    load_side_borders, get_neutron_window_paths)
from data_processing.helpers import stop

In [ ]:
CalibrationKey = Literal[ExperimentDataKey.CAEN_CALIBRATION, ExperimentDataKey.NEW_CALIBRATION]
NasaBorderKey = Literal[ExperimentDataKey.NASA_BORDERS, ExperimentDataKey.NASA_BORDERS_RECALC]

### Functions

In [ ]:
def bin_non_neutron_data(df, time_bins, data_col, selected_cols):
    start_time = time_bins[0]
    df = get_time_cut(df, 'Time', time_bins)

    binned_df = df.groupby("Time Bin", as_index=False)[data_col] \
        .agg(['mean', 'std']) \
        .copy()
    binned_df.columns = selected_cols
    binned_df['Bin midpoint'] = binned_df.index.to_series() \
        .apply(lambda x: x.mid)
    binned_df = bin_midpoint_time_to_seconds(binned_df, start_time)

    return binned_df

In [ ]:
def bin_midpoint_time_to_seconds(df, start_time):
    bin_mid_col = df[BinningDataframeColumn.BIN_MIDPOINT.value]
    bin_time_col_name = BinningDataframeColumn.BIN_TIME.value
    zeroed_midpoint = pd.to_datetime(bin_mid_col) - start_time
    df[bin_time_col_name] = zeroed_midpoint.dt.total_seconds()
    return df

In [ ]:
def get_time_cut(df, time_tag_col, time_bins):
    timetag_cut = pd.cut(df[time_tag_col], bins=time_bins)
    df[BinningDataframeColumn.TIME_BIN.value] = timetag_cut
    return df

In [ ]:
# fns ask questions, then generate strategy using factory

CalibrationKey = Literal[ExperimentDataKey.CAEN_CALIBRATION, ExperimentDataKey.NEW_CALIBRATION]
NasaBorderKey = Literal[ExperimentDataKey.NASA_BORDERS, ExperimentDataKey.NASA_BORDERS_RECALC]


def get_nasa_loading_settings(
    calib_key: CalibrationKey
) -> str:
    left_border_type = helpers.get_input_with_default(
        """\
Which left border calculation do you want to use?
1: original left border (0.1966 MeVee)
2: newer left border (~0.1866 MeVee)
3: CAEN lower limit (0.050 MeVee) (default)
Press Enter for default
""",
        3,
        int
    )
    border_key: NasaBorderKey = (
        ExperimentDataKey.NASA_BORDERS if left_border_type == 1 
        else ExperimentDataKey.NASA_BORDERS_RECALC
    )
    file_name_prefix = f"{calib_key.value}_{border_key.value}"
    return file_name_prefix


def get_n_distro_loading_settings(
    calib_key: CalibrationKey
) -> str:
    file_name_prefix = f"{calib_key.value}_{ExperimentDataKey.N_WINDOW_BORDERS.value}"
    return file_name_prefix


def get_nasa_generation_settings(
    calib_key: CalibrationKey
) -> NasaGenerationSettings:
    sigma = helpers.get_input_with_default(
        """\
Enter value of sigma
Press Enter for default (5)
""",
        5,
        float
    )
    window_offset = helpers.get_input_with_default(
        """\
Enter value of offset between top and bottom window border
Press Enter for default (0.2)
""",
        0.2,
        float
    )
    left_border_type_input = helpers.get_input_with_default(
        """\
How do you want to handle the left border?
1: use existing value (default)
2: recalculate from data
3: enter own value
Press Enter for default
""",
        1,
        int
    )
    if left_border_type_input == 1:
        existing_left_border_version_input = helpers.get_input_with_default(
            """\
Which existing left border do you want to use?
1: original (0.1966 MeVee)
2: newer (~0.1866 MeVee)
3: detector lower limit (0.050 MeVee) (default)
or press Enter for default
""",
            3,
            int
        )
        if existing_left_border_version_input in [1, 2]:
            border_key = (
                ExperimentDataKey.NASA_BORDERS 
                if existing_left_border_version_input == 1 
                else ExperimentDataKey.NASA_BORDERS_RECALC
            )
            file_name_prefix = f"{calib_key.value}_{border_key.value}"
            side_borders_path, *_ = get_neutron_window_paths(
                file_name_prefix=file_name_prefix)
            left_border, _ = load_side_borders(
                side_borders_path=side_borders_path)
            if left_border is None:
                raise ValueError("Left border could not be loaded")
            lower_energy_bound = left_border
            recalc_lower_bound = False
        elif existing_left_border_version_input == 3:
            lower_energy_bound = 0.05
            recalc_lower_bound = False
        else:
            raise ValueError("Unsupported choice")
        pass
    elif left_border_type_input == 2:
        lower_energy_bound = 0.1966
        recalc_lower_bound = True
    elif left_border_type_input == 3:
        lower_energy_bound = helpers.get_input_with_default(
            """\
Enter value of lower energy bound (in MeVee)
Press Enter for default (0.050)
""",
            0.050,
            float
        )
        recalc_lower_bound = False
    else:
        raise ValueError("Unsupported choice")
    settings = NasaGenerationSettings(
        window_offset=window_offset,
        sigma=sigma,
        lower_energy_bound=lower_energy_bound,
        recalculate_lower_energy_bound=recalc_lower_bound
    )
    return settings


def get_n_distro_generation_settings(
) -> NeutronDistributionGenerationSettings:
    sigma = helpers.get_input_with_default(
        """\
Enter value of sigma
Press Enter for default (3)
""",
        3,
        float
    )
    settings = NeutronDistributionGenerationSettings(
        sigma=sigma
    )
    return settings


def make_strategy_factory_fn(
    strategy_factory: NeutronStrategyFactory,
    window_type: WindowType,
    loading: bool,
    settings: NeutronWindowSettings
) -> Callable[[], AbstractNeutronStrategy]:
    def factory_fn():
        return strategy_factory.make_neutron_window_strategy(
            window_type, loading, settings
        )
    return factory_fn


def make_strategy_for_experiments(
    experiment_neutron_data: ExperimentNeutronData, 
    factory_fn: Callable[[], AbstractNeutronStrategy]
) -> ExperimentNeutronData:
    new_neutron_data = {
        exp_id: {**exp_data, ExperimentDataKey.BORDER_STRATEGY: factory_fn()}
        for exp_id, exp_data
        in experiment_neutron_data.items()
    }
    return new_neutron_data

In [ ]:
def get_psd_adc_histogram(
    df: pd.DataFrame,
    adc_width: float = 420,
    adc_bins: np.ndarray | None = None,
    psd_bin_count: int = 100,
    psd_min: float = 0.0,
    psd_max: float = 0.5
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    x = get_df_col(df, DetectorDataframeColumn.ENERGY)
    y = get_df_col(df, DetectorDataframeColumn.PSD)

    within_psd = y.between(psd_min, psd_max)
    x = x[within_psd == True].copy()
    y = y[within_psd == True].copy()

    if adc_bins is not None:
        x_bins = adc_bins
    else:
        x_bins: np.ndarray = np.linspace(
            0, x.max(), int(x.max() / adc_width) + 1
        )
    print(f"Energy width = {x_bins[1]-x_bins[0]} ADC")
    y_bins: np.ndarray = np.linspace(psd_min, psd_max, psd_bin_count + 1)

    Z, xe, ye = np.histogram2d(x, y, bins=[x_bins, y_bins])
    return Z, xe, ye

In [ ]:
def lerp(x: float, p1: tuple[float, float], p2: tuple[float, float]) -> float:
    x1, y1 = p1
    x2, y2 = p2
    m = (y2 - y1) / (x2 - x1)
    return m * (x - x1) + y1


def clamp(x: float, xmin: float, xmax: float) -> float:
    if x < xmin:
        return xmin
    elif x > xmax:
        return xmax
    else:
        return x


def to_unit_interval(x: float, xmin: float, xmax: float) -> float:
    return clamp(lerp(x, (xmin, 0), (xmax, 1)), 0, 1)


ColorHexCode = str  # of format #xxxxxx, can be made into hex value
ColorAlphaHexCode = str  # of format #xxxxxxxx, can be made into hex value
ColorTuple = tuple[float, float, float]
ColorAlphaTuple = tuple[float, float, float, float]
ColorType = ColorHexCode | ColorAlphaHexCode | ColorTuple | ColorAlphaTuple
CT = TypeVar("CT", bound=ColorType)


def is_color_hex_code(x: Any) -> TypeGuard[ColorHexCode]:
    if not isinstance(x, str):
        return False
    pattern_match = re.match(r"#[0-9a-fA-F]{6}$", x)
    return pattern_match is not None


def is_color_alpha_hex_code(x: Any) -> TypeGuard[ColorAlphaHexCode]:
    if not isinstance(x, str):
        return False
    pattern_match = re.match(r"#[0-9a-fA-F]{8}$", x)
    return pattern_match is not None


def is_color_tuple(x: Any) -> TypeGuard[ColorTuple]:
    if not isinstance(x, tuple):
        return False
    if not len(x) == 3:
        return False
    elements_in_limits = [0 <= elem <= 1 for elem in x]
    return all(elements_in_limits)


def is_color_alpha_tuple(x: Any) -> TypeGuard[ColorAlphaTuple]:
    if not isinstance(x, tuple):
        return False
    if not len(x) == 4:
        return False
    elements_in_limits = [0 <= elem <= 1 for elem in x]
    return all(elements_in_limits)


def to_color_float(x: str) -> float:
    try:
        color = int(x, base=16) / 255
    except ValueError:
        raise ValueError(f"{x} is not valid hexadecimal code")
    if color < 0 or color > 1:
        raise ValueError(f"{x} is out of color float bounds (0-1)")
    return color


def to_hex_chars(x: float) -> str:
    if x < 0 or x > 1:
        raise ValueError(f"{x} is out of color float bounds (0-1)")
    x_int = min(255, int(x * 256))
    return f"{x_int:02X}"


def to_color_alpha_tuple(x: ColorType) -> ColorAlphaTuple:
    if is_color_alpha_tuple(x):
        return x
    elif is_color_alpha_hex_code(x):
        r = to_color_float(x[1:3])
        g = to_color_float(x[3:5])
        b = to_color_float(x[5:7])
        a = to_color_float(x[7:9])
        return (r, g, b, a)
    elif is_color_tuple(x):
        r, g, b = x
        return (r, g, b, 1)
    elif is_color_hex_code(x):
        r = to_color_float(x[1:3])
        g = to_color_float(x[3:5])
        b = to_color_float(x[5:7])
        return (r, g, b, 1)
    else:
        raise ValueError(f"{x} could not be converted to ColorAlphaTuple")


def to_color_tuple(x: ColorType) -> ColorTuple:
    if is_color_alpha_tuple(x):
        r, g, b, _ = x
        return (r, g, b)
    elif is_color_alpha_hex_code(x):
        r = to_color_float(x[1:3])
        g = to_color_float(x[3:5])
        b = to_color_float(x[5:7])
        return (r, g, b)
    elif is_color_tuple(x):
        return x
    elif is_color_hex_code(x):
        r = to_color_float(x[1:3])
        g = to_color_float(x[3:5])
        b = to_color_float(x[5:7])
        return (r, g, b)
    else:
        raise ValueError(f"{x} could not be converted to ColorTuple")


def to_color_hex_code(x: ColorType) -> ColorHexCode:
    if is_color_alpha_tuple(x):
        r, g, b, _ = x
        r_hex = to_hex_chars(r)
        g_hex = to_hex_chars(g)
        b_hex = to_hex_chars(b)
        return ("#" + r_hex + g_hex + b_hex).lower()
    elif is_color_alpha_hex_code(x):
        return x[:-2]
    elif is_color_tuple(x):
        r, g, b = x
        r_hex = to_hex_chars(r)
        g_hex = to_hex_chars(g)
        b_hex = to_hex_chars(b)
        return ("#" + r_hex + g_hex + b_hex).lower()
    elif is_color_hex_code(x):
        return x
    else:
        raise ValueError(f"{x} could not be converted to ColorHexCode")


def to_color_alpha_hex_code(x: ColorType) -> ColorAlphaHexCode:
    if is_color_alpha_tuple(x):
        r, g, b, a = x
        r_hex = to_hex_chars(r)
        g_hex = to_hex_chars(g)
        b_hex = to_hex_chars(b)
        a_hex = to_hex_chars(a)
        return ("#" + r_hex + g_hex + b_hex + a_hex).lower()
    elif is_color_alpha_hex_code(x):
        return x
    elif is_color_tuple(x):
        r, g, b = x
        r_hex = to_hex_chars(r)
        g_hex = to_hex_chars(g)
        b_hex = to_hex_chars(b)
        return ("#" + r_hex + g_hex + b_hex + "ff").lower()
    elif is_color_hex_code(x):
        return x+"ff"
    else:
        raise ValueError(f"{x} could not be converted to ColorAlphaHexCode")


def calculate_gradient_color(
    gradient_unit_interval: float,
    color_from: ColorAlphaTuple,
    color_to: ColorAlphaTuple
) -> ColorAlphaTuple:
    if gradient_unit_interval < 0 or gradient_unit_interval > 1:
        raise ValueError(f"{x} is out of bounds (0-1)")
    grad_color = tuple([
        clamp(lerp(gradient_unit_interval, (0, val_from), (1, val_to)), val_from, val_to)
        for val_from, val_to in zip(color_from, color_to)
    ])
    return grad_color

## Experiment ID Input

In [ ]:
experiment_ids = ["TB-26"]

In [ ]:
# more here?
experiment_neutron_data: ExperimentNeutronData = {
    exp_id: {}
    for exp_id in experiment_ids
}

In [ ]:
calib_input = "y"

is_new_calibration = calib_input.lower() == "y"
calibrated_energy_column: EnergyColumn = (
    DetectorDataframeColumn.RECALIBRATED_ENERGY
    if is_new_calibration else DetectorDataframeColumn.CALIB_ENERGY
)
calib_key: CalibrationKey = ExperimentDataKey.NEW_CALIBRATION if is_new_calibration else ExperimentDataKey.CAEN_CALIBRATION

In [ ]:
detector_code = proc.Detector.ONE

In [ ]:
fit_style = "peak_finder"

In [ ]:
strategy_factory = proc.NeutronStrategyFactory()
window_offset = 0.2
sigma = 5
lower_energy_bound = 0.05
recalc_lower_bound = False

calib_params = DetectorCalibrationParams[detector_code]
lower_energy_bound = 1000 * calib_params.p1 * lower_energy_bound + calib_params.p2

settings = proc_types.NasaGenerationSettings(
        window_offset=window_offset,
        sigma=sigma,
        lower_energy_bound=lower_energy_bound,
        recalculate_lower_energy_bound=recalc_lower_bound
    )
factory_fn = make_strategy_factory_fn(
    strategy_factory, "nasa", False, settings)
experiment_neutron_data = make_strategy_for_experiments(
    experiment_neutron_data, factory_fn)

In [ ]:
bin_length = 300
bin_string = f"{bin_length}s"

## Data Loading and Initial Processing

### Data Loading

In [ ]:
# Data Loading
for exp_id, exp_data in experiment_neutron_data.items():
    exp_data[ExperimentDataKey.UNCLASSIFIED] = load_parquet_psd(exp_id)

### Initial Processing

In [ ]:
# Express timetags in hours elapsed
for exp_id, exp_data in experiment_neutron_data.items():
    unclassified_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
    unclassified_df = calculate_timetag_hours(unclassified_df)
    exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df

In [ ]:
# Recalibrate energy
for exp_id, exp_data in experiment_neutron_data.items():
    unclassified_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
    unclassified_df = proc.recalibrate(unclassified_df, detector_code)
    exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df

In [ ]:
# Generate histogram

start_scan_idx = 0
end_scan_idx = 420
adc_width = 20

for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED]
    Z, xe, ye = get_psd_adc_histogram(
        psd_report,
        adc_width=adc_width,
    )
    exp_data[ExperimentDataKey.PSD_HISTOGRAM] = Z
    exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES] = xe
    exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES] = ye
    exp_data[ExperimentDataKey.END_SCAN_IDX] = min(end_scan_idx, len(Z))

## Neutron Classification

In [ ]:
# TODO get fit dataframe (not needed if loading, but do anyway to keep process consistent)
stop_here = False

for exp_id, exp_data in experiment_neutron_data.items():
    Z = exp_data[ExperimentDataKey.PSD_HISTOGRAM]
    xe = exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES]
    ye = exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES]
    end_scan_idx = exp_data[ExperimentDataKey.END_SCAN_IDX]

    df, df_err = proc.scan_histogram_slices(
        Z,
        xe,
        ye,
        fit_style="peak_finder",
        start_idx=start_scan_idx,
        end_idx=end_scan_idx
    )
    df, bad_slice_indexes = proc.find_failed_slices(df, exp_id)

    if bad_slice_indexes is not None:
        exp_data[ExperimentDataKey.VALID_SLICE_FITS] = df
        exp_data[ExperimentDataKey.BAD_SLICE_INDEXES] = bad_slice_indexes
        stop_here = True
    else:
        exp_data[ExperimentDataKey.FOM_RESULTS] = df

if stop_here:
    stop()

In [ ]:
# get borders from strategy
for exp_id, exp_data in experiment_neutron_data.items():
    if ExperimentDataKey.FOM_RESULTS not in exp_data:
        print(f"No good fit data on Experiment {exp_id}")
        continue

    fom_results = exp_data[ExperimentDataKey.FOM_RESULTS]
    strategy = exp_data[ExperimentDataKey.BORDER_STRATEGY]

    strategy.set_slice_fit_dataframe(fom_results)
    borders = strategy.get_neutron_window()

    exp_data[ExperimentDataKey.BORDERS] = borders

In [ ]:
# classify neutrons
for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED].copy()
    borders = exp_data[ExperimentDataKey.BORDERS]

    psd_report = proc.classify(
        psd_report,
        DetectorDataframeColumn.ENERGY,
        borders,
        DetectorDataframeColumn.NEW_N_CLASS
    )

    exp_data[ExperimentDataKey.PSD_REPORT] = psd_report

In [ ]:
# Get experiment start time
for exp_name, data_dict in experiment_neutron_data.items():
    exp_root = get_exp_root(exp_name)
    with open(exp_root / 'exp_info.toml') as exp_info:
        exp_start_line = [line for line in exp_info if "exp_start" in line][0]
    exp_start_text = exp_start_line.replace("exp_start = ", "").strip()
    exp_start = datetime.fromisoformat(exp_start_text).astimezone(timezone.utc)
    data_dict[ExperimentDataKey.START_TIME] = exp_start

In [ ]:
# Get timetag as clock time
for exp_name, data_dict in experiment_neutron_data.items():
    psd_report = data_dict[ExperimentDataKey.PSD_REPORT]
    exp_start = data_dict[ExperimentDataKey.START_TIME]

    psd_report = calculate_event_time(psd_report, exp_start)

    data_dict[ExperimentDataKey.PSD_REPORT] = psd_report

In [ ]:
# Separate neutron and gamma events
for exp_name, data_dict in experiment_neutron_data.items():
    psd_report = data_dict[ExperimentDataKey.PSD_REPORT]

    n_classify_col_name = DetectorDataframeColumn.NEW_N_CLASS.value
    neutrons_only = psd_report.query(n_classify_col_name).copy()
    gamma_only = psd_report.query(f"~{n_classify_col_name}").copy()
    data_dict[ExperimentDataKey.NEUTRONS_ONLY] = neutrons_only
    data_dict[ExperimentDataKey.GAMMA_ONLY] = gamma_only

## Data Binning

In [ ]:
# Create bins
for exp_name, data_dict in experiment_neutron_data.items():
    print(exp_name)
    neutron_report = data_dict[ExperimentDataKey.NEUTRONS_ONLY]

    event_time_col = DetectorDataframeColumn.EVENT_TIME.value
    start_time = neutron_report[event_time_col].min()
    end_time = neutron_report[event_time_col].max()
    print(neutron_report[event_time_col])
    timetag_clock_bins = pd.date_range(
        start=start_time, end=end_time, freq=bin_string)
    data_dict[ExperimentDataKey.TIME_BIN_EDGES] = timetag_clock_bins

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.PSD_REPORT]
    time_bin_edges = exp_data[ExperimentDataKey.TIME_BIN_EDGES]

    time_bin_histogram_data = {}
    bin_idxs = [0, 2]
    for bin_index in bin_idxs:
        low_time_edge = time_bin_edges[bin_index]
        high_time_edge = time_bin_edges[bin_index + 1]
        event_time_col = DetectorDataframeColumn.EVENT_TIME.value
        time_bin_df = psd_report[
            psd_report[event_time_col].between(low_time_edge, high_time_edge)
        ]

        Z, xe, ye = get_psd_adc_histogram(
            time_bin_df,
            adc_width=adc_width
        )
        time_bin_histogram_data[bin_index] = {
            "histogram": Z,
            "energy_edges": xe,
            "psd_edges": ye
        }
    exp_data["time_bin_histogram"] = time_bin_histogram_data

In [ ]:
# Bin neutron data
for exp_name, data_dict in experiment_neutron_data.items():
    neutron_report = data_dict[ExperimentDataKey.NEUTRONS_ONLY]
    time_bins = data_dict[ExperimentDataKey.TIME_BIN_EDGES]

    time_col_name = DetectorDataframeColumn.EVENT_TIME.value
    time_bin_col_name = BinningDataframeColumn.TIME_BIN.value
    count_col_name = BinningDataframeColumn.COUNT.value
    count_error_col_name = BinningDataframeColumn.COUNT_ERROR.value
    bin_mid_col_name = BinningDataframeColumn.BIN_MIDPOINT.value
    n_rate_col_name = BinningDataframeColumn.NEUTRON_RATE.value
    n_error_col_name = BinningDataframeColumn.NEUTRON_RATE_ERROR.value

    start_time = time_bins[0]

    binned_neutrons = get_time_cut(
        neutron_report, time_col_name, time_bins)
    binned_neutrons = neutron_report.groupby(
        time_bin_col_name, as_index=True, observed=False) \
        .size() \
        .to_frame() \
        .copy()
    binned_neutrons.columns = [count_col_name]
    binned_neutrons[count_error_col_name] = np.sqrt(
        binned_neutrons[count_col_name]
    )

    binned_neutron_time_bins = binned_neutrons.index.to_series()
    midpoints = binned_neutron_time_bins.apply(lambda x: x.mid)
    durations = binned_neutron_time_bins.apply(
        lambda x: x.length.total_seconds()
    ).astype(np.float64)

    binned_neutrons[bin_mid_col_name] = midpoints
    binned_neutrons = bin_midpoint_time_to_seconds(binned_neutrons, start_time)

    binned_neutrons[n_rate_col_name] = (
        binned_neutrons[count_col_name] / durations)
    binned_neutrons[n_error_col_name] = (
        binned_neutrons[count_error_col_name] / durations)
    binned_neutrons = binned_neutrons.drop(
        [count_col_name, count_error_col_name],
        axis=1
    ) \
        .copy()
    data_dict[ExperimentDataKey.BINNED_NEUTRONS] = binned_neutrons

## Figure Base Data

In [ ]:
dl_folder = Path.home() / "Downloads" / "neutron_detection_paper"
dl_folder.mkdir(parents=True, exist_ok=True)

In [ ]:
def get_figure_4a_data(
    e_margin: float = 2,
    psd_margin: float = 0.002
) -> pd.DataFrame:
    """
    Contains 2D histogram of neutron detector data
    Each row represents one bar in the histogram
    The columns are the parameters needed to place that bar in the 3D plot
    Also included are the bar indices (in the "xi" and "yi" columns)
    """
    exp_name = "TB-26"
    data_dict = experiment_neutron_data[exp_name]
    xe = data_dict[ExperimentDataKey.HISTOGRAM_X_EDGES]
    ye = data_dict[ExperimentDataKey.HISTOGRAM_Y_EDGES]
    dz = data_dict[ExperimentDataKey.PSD_HISTOGRAM]

    x, y = np.meshgrid(xe[:-1], ye[:-1])
    x, y = x.ravel(), y.ravel()
    xi, yi = np.meshgrid(range(len(xe)-1), range(len(ye)-1))
    xi, yi = xi.ravel(), yi.ravel()
    z = np.full_like(x, 0)
    _dx = xe[1:] - xe[:-1]
    _dy = ye[1:] - ye[:-1]
    dx, dy = np.meshgrid(_dx, _dy)
    dz = dz.T
    dx, dy, dz = dx.ravel(), dy.ravel(), dz.ravel()

    x = x + e_margin
    dx = dx - e_margin
    y = y + psd_margin
    dy = dy - psd_margin

    height_mask = dz > 10
    x = x[height_mask]
    y = y[height_mask]
    xi = xi[height_mask]
    yi = yi[height_mask]
    z = z[height_mask]
    dx = dx[height_mask]
    dy = dy[height_mask]
    dz = dz[height_mask]

    df = pd.DataFrame(
        data={
            "x": x, "y": y, "z": z,
            "xi": xi, "yi": yi,
            "dx": dx, "dy": dy, "dz": dz
        }
    )
    return df

In [ ]:
def get_figure_4b_data() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    First DataFrame contains a chosen slice of the 2D histogram
    This slice is for a specific light energy bin in the histogram
    The index is the midpoint of the PSD bin, and the series value is the bin count
    Second DataFrame contains data for plotting best fit gaussian curves
    Two series are provided for the gamma ray and neutron "channels" in the detector data
    The index is the PSD value, and the series values are the fit curve values
    Third DataFrame contains "marker values" for each fit curve
    These values are the curve mean ("mu") and outer curve boundary of mu + 5*sigma ("border")
    """
    exp_data = experiment_neutron_data["TB-26"]
    fom_df = exp_data[ExperimentDataKey.FOM_RESULTS]
    Z = exp_data[ExperimentDataKey.PSD_HISTOGRAM]
    ye = exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES]

    fom_slice = fom_df.iloc[chosen_slice_idx]
    mu1, sigma1, a1 = [fom_slice[val_name] for val_name in ["mu1", "sigma1", "a1"]]
    mu2, sigma2, a2 = [fom_slice[val_name] for val_name in ["mu2", "sigma2", "a2"]]
    Z_slice = Z[chosen_slice_idx, :]
    ymids = (ye[:-1] + ye[1:]) / 2

    gauss_y = np.linspace(0, 0.6, 1000)
    gamma_gaussian = proc.gaussian(gauss_y, mu1, sigma1, a1)
    neutron_gaussian = proc.gaussian(gauss_y, mu2, sigma2, a2)
    gamma_vline_xs = [mu1, mu1 + 5 * sigma1]
    neutron_vline_xs = [mu2, mu2 + 5 * sigma2]

    gaussian_df = pd.DataFrame(
        data={"neutron": neutron_gaussian, "gamma": gamma_gaussian},
        index=gauss_y
    )
    slice_df = pd.DataFrame(
        data={"count": Z_slice},
        index=ymids
    )
    vline_df = pd.DataFrame(
        data={"neutron": neutron_vline_xs, "gamma": gamma_vline_xs},
        index=["mu", "border"]
    )

    return slice_df, gaussian_df, vline_df

In [ ]:
def get_figure_4c_data(
    e_margin: float = 2,
    psd_margin: float = 0.002
) -> pd.DataFrame:
    """
    Contains 2D histogram of neutron detector data
    Each row represents one bar in the histogram
    The columns are the parameters needed to place that bar in the 3D plot
    Also included is whether each bar is within the neutron window borders
    """
    data_dict = experiment_neutron_data["TB-26"]
    xe = data_dict[ExperimentDataKey.HISTOGRAM_X_EDGES]
    ye = data_dict[ExperimentDataKey.HISTOGRAM_Y_EDGES]
    dz = data_dict[ExperimentDataKey.PSD_HISTOGRAM]
    borders = data_dict[ExperimentDataKey.BORDERS]

    x, y = np.meshgrid(xe[:-1], ye[:-1])
    x, y = x.ravel(), y.ravel()
    z = np.full_like(x, 0)
    _dx = xe[1:] - xe[:-1]
    _dy = ye[1:] - ye[:-1]
    dx, dy = np.meshgrid(_dx, _dy)
    dz = dz.T
    dx, dy, dz = dx.ravel(), dy.ravel(), dz.ravel()

    _xmids = (xe[:-1] + xe[1:]) / 2
    _ymids = (ye[:-1] + ye[1:]) / 2
    xmids, ymids = np.meshgrid(_xmids, _ymids)
    xmids, ymids = xmids.ravel(), ymids.ravel()
    if borders.left is None:
        within_left_border = np.full_like(xmids, True, dtype=bool)
    else:
        within_left_border = xmids >= borders.left
    if borders.right is None:
        within_right_border = np.full_like(xmids, True, dtype=bool)
    else:
        within_right_border = xmids <= borders.right
    if borders.bottom is None:
        within_bottom_border = np.full_like(ymids, True, dtype=bool)
    else:
        bottom_border_psds = borders.bottom(xmids)
        within_bottom_border = ymids >= bottom_border_psds
    if borders.top is None:
        within_top_border = np.full_like(ymids, True, dtype=bool)
    else:
        top_border_psds = borders.top(xmids)
        within_top_border = ymids <= top_border_psds
    within_borders = (within_left_border &
                      within_right_border &
                      within_bottom_border &
                      within_top_border)

    x = x + e_margin
    dx = dx - e_margin
    y = y + psd_margin
    dy = dy - psd_margin

    height_mask = dz > 10
    x = x[height_mask]
    y = y[height_mask]
    z = z[height_mask]
    dx = dx[height_mask]
    dy = dy[height_mask]
    dz = dz[height_mask]
    within_borders = within_borders[height_mask]

    df = pd.DataFrame(
        data={
            "x": x,
            "y": y,
            "z": z,
            "dx": dx,
            "dy": dy,
            "dz": dz,
            "within_borders": within_borders
        }
    )
    return df

In [ ]:
def get_figure_4d_main_data() -> pd.DataFrame:
    """
    Contains neutron rate data over time
    The index is the midpoint of each time bin in minutes
    The series value is the neutron rate in that bin (in neutrons per second)
    """
    data_dict = experiment_neutron_data["TB-26"]

    binned_neutrons = data_dict[ExperimentDataKey.BINNED_NEUTRONS]

    bin_time_col_name = BinningDataframeColumn.BIN_TIME.value
    n_rate_col_name = BinningDataframeColumn.NEUTRON_RATE.value
    zeroed_bins = binned_neutrons[bin_time_col_name] / 60
    rates = binned_neutrons[n_rate_col_name]

    df = pd.DataFrame(
        data={"time": zeroed_bins, "rate": rates}
    )
    df = df.set_index("time")
    return df

In [ ]:
def get_figure_4d_side_data(
    bin_idx: int,
    e_margin: float = 2,
    psd_margin: float = 0.002
) -> pd.DataFrame:
    """
    Contains 2D histogram of neutron detector data for a specific time bin
    Each row represents one bar in the histogram
    The columns are the parameters needed to place that bar in the 3D plot
    Also included is whether each bar is within the neutron window borders
    """
    exp_data = experiment_neutron_data["TB-26"]
    histo_data = exp_data["time_bin_histogram"][bin_idx]
    xe = histo_data["energy_edges"]
    ye = histo_data["psd_edges"]
    dz = histo_data["histogram"]

    x, y = np.meshgrid(xe[:-1], ye[:-1])
    x, y = x.ravel(), y.ravel()
    z = np.full_like(x, 0)
    _dx = xe[1:] - xe[:-1]
    _dy = ye[1:] - ye[:-1]
    dx, dy = np.meshgrid(_dx, _dy)
    dz = dz.T
    dx, dy, dz = dx.ravel(), dy.ravel(), dz.ravel()

    _xmids = (xe[:-1] + xe[1:]) / 2
    _ymids = (ye[:-1] + ye[1:]) / 2
    xmids, ymids = np.meshgrid(_xmids, _ymids)
    xmids, ymids = xmids.ravel(), ymids.ravel()
    if borders.left is None:
        within_left_border = np.full_like(xmids, True, dtype=bool)
    else:
        within_left_border = xmids >= borders.left
    if borders.right is None:
        within_right_border = np.full_like(xmids, True, dtype=bool)
    else:
        within_right_border = xmids <= borders.right
    if borders.bottom is None:
        within_bottom_border = np.full_like(ymids, True, dtype=bool)
    else:
        bottom_border_psds = borders.bottom(xmids)
        within_bottom_border = ymids >= bottom_border_psds
    if borders.top is None:
        within_top_border = np.full_like(ymids, True, dtype=bool)
    else:
        top_border_psds = borders.top(xmids)
        within_top_border = ymids <= top_border_psds
    within_borders = (within_left_border &
                      within_right_border &
                      within_bottom_border &
                      within_top_border)

    x = x + e_margin
    dx = dx - e_margin
    y = y + psd_margin
    dy = dy - psd_margin

    height_mask = dz > 1
    x = x[height_mask]
    y = y[height_mask]
    z = z[height_mask]
    dx = dx[height_mask]
    dy = dy[height_mask]
    dz = dz[height_mask]
    within_borders = within_borders[height_mask]

    df = pd.DataFrame(
        data={
            "x": x,
            "y": y,
            "z": z,
            "dx": dx,
            "dy": dy,
            "dz": dz,
            "within_borders": within_borders
        }
    )
    return df

## Plotting

### Plot Style Constants

In [ ]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial'] + plt.rcParams['font.sans-serif']

fontsize = 7
fontsize_small = 5
linewidth = 1
plt.rcParams['font.size'] = fontsize
plt.rcParams['lines.linewidth'] = linewidth

In [ ]:
bg_blue = "#4c94ff"
bg_red = "#f54336"
bg_grey = "#9e9e9e"
bg_bluegrey = "#8a9fb8"

In [ ]:
chosen_slice_idx = 12

### Plot Functions

#### Plot Helpers

In [ ]:
def set_3d_ticklabel_alignment(ax: mpl.axes.Axes):
    xaxis_ticklabels = ax.xaxis.get_ticklabels()
    for ticklabel in xaxis_ticklabels:
        ticklabel.set_ha("right")
        ticklabel.set_va("baseline")
    yaxis_ticklabels = ax.yaxis.get_ticklabels()
    for ticklabel in yaxis_ticklabels:
        ticklabel.set_ha("center")
        ticklabel.set_va("top")
    zaxis_ticklabels = ax.zaxis.get_ticklabels()
    for ticklabel in zaxis_ticklabels:
        ticklabel.set_ha("left")
        ticklabel.set_va("center_baseline")

In [ ]:
def set_3d_ticklabel_padding(
    ax: mpl.axes.Axes, x_pad: float, y_pad: float, z_pad: float
):
    # all padding is relative to fontsize
    ax.tick_params(axis="x", pad=x_pad*fontsize)
    ax.tick_params(axis="y", pad=y_pad*fontsize)
    ax.tick_params(axis="z", pad=z_pad*fontsize)

In [ ]:
def set_3d_axis_label_padding(
    ax: mpl.axes.Axes, x_pad: float, y_pad: float, z_pad: float
):
    ax.xaxis.labelpad = x_pad * fontsize
    ax.yaxis.labelpad = y_pad * fontsize
    ax.zaxis.labelpad = z_pad * fontsize

In [ ]:
def set_3d_panes_transparent(ax: mpl.axes.Axes):
    ax.xaxis.set_pane_color((1, 1, 1, 0))
    ax.yaxis.set_pane_color((1, 1, 1, 0))
    ax.zaxis.set_pane_color((1, 1, 1, 0))

#### Main Functions

In [ ]:
def plot_figure_4a(ax: mpl.axes.Axes):
    cmap = plt.colormaps["viridis"]
    # fontsize_offset = -6
    angle_elev = 30
    angle_rot = -20
    e_margin = 2
    psd_margin = 0.002

    df = get_figure_4a_data(e_margin, psd_margin)
    x = df["x"]
    y = df["y"]
    z = df["z"]
    xi = df["xi"]
    dx = df["dx"]
    dy = df["dy"]
    dz = df["dz"]

    min_dz = -2000
    max_dz = np.max(dz)
    norm = mpl.colors.Normalize(vmin=min_dz, vmax=max_dz)
    mapped_colors = [cmap(norm(dz_val)) for dz_val in dz]
    is_chosen_slice = xi == chosen_slice_idx
    mapped_colors = [
        color if is_chosen else (color[0], color[1], color[2], 0.02)
        for is_chosen, color
        in zip(is_chosen_slice, mapped_colors)
    ]

    ax.view_init(angle_elev, angle_rot)
    ax.bar3d(
        x, y, z, dx, dy, dz,
        color=mapped_colors,
        shade=False,
        zsort="max",
    )

    # Blur window
    window_x = xe[chosen_slice_idx]
    bg_grey_alpha = to_color_alpha_tuple(bg_grey)
    bg_grey_alpha = (bg_grey_alpha[0], bg_grey_alpha[1], bg_grey_alpha[2], 0.2)
    window = mpl.patches.Rectangle((0, 0), 0.5, 4000, fc=bg_grey_alpha, ec="black", zorder=0)
    ax.add_patch(window)
    art3d.pathpatch_2d_to_3d(window, z=window_x, zdir="x")

    ax.set_xlabel("Energy (ADC channel x1000)")
    ax.set_ylabel("PSD")
    ax.set_zlabel("Counts (x1000)")
    set_3d_axis_label_padding(ax, -1.3, -1.2, -1.5)
    
    ax.set_xlim(0, 3000)
    ax.set_ylim(0, 0.5)
    ax.set_zlim(0, 4000)
    
    ax.xaxis.set_major_formatter(lambda x, pos: f"{x / 1000:.1f}")
    ax.zaxis.set_major_formatter(lambda z, pos: f"{z / 1000:.1f}")
    ax.tick_params(labelsize=fontsize_small)

    set_3d_ticklabel_padding(ax, -0.8, -0.7, -1.0)
    set_3d_ticklabel_alignment(ax)

    set_3d_panes_transparent(ax)

In [ ]:
def plot_figure_4b(ax: mpl.axes.Axes):
    slice_df, gaussian_df, vline_df = get_figure_4b_data()
    gauss_x = gaussian_df.index
    gauss_gamma = gaussian_df["gamma"]
    gauss_neutron = gaussian_df["neutron"]

    gamma_vline_series = vline_df["gamma"]
    neutron_vline_series = vline_df["neutron"]
    vline_series = pd.concat([gamma_vline_series, neutron_vline_series])

    gamma_mu = gamma_vline_series["mu"]
    gamma_border = gamma_vline_series["border"]
    neutron_mu = neutron_vline_series["mu"]
    neutron_border = neutron_vline_series["border"]

    ax.plot(gauss_x, gauss_gamma, color=bg_grey)
    ax.plot(gauss_x, gauss_neutron, color=bg_grey)
    ax.plot(slice_df.index, slice_df["count"], "-", color="black", alpha=0.5)

    # Gaussian feature lines
    ax.axvline(0, 0, 3000, alpha=0)  # "burner" line (explained below)
    # first line never transforms properly (transform not initiated?)
    for vline_x in vline_series:
        ax.axvline(vline_x, 0, 1, color="black")
    margin = 0.005
    ax.text(
        gamma_mu - margin,
        3100,
        r"${\mu}_{\gamma}$",
        fontsize=fontsize_small,
        ha="right"
    )
    ax.text(
        gamma_border - margin,
        3100,
        r"${\mu}_{\gamma} + 5{\sigma}_{\gamma}$",
        fontsize=fontsize_small,
        ha="right"
    )
    ax.text(
        neutron_mu - margin,
        3100,
        r"${\mu}_{n}$",
        fontsize=fontsize_small,
        ha="right"
    )
    ax.text(
        neutron_border - margin,
        3100,
        r"${\mu}_{n} + 5{\sigma}_{n}$",
        fontsize=fontsize_small,
        ha="right"
    )

    # Color gamma/neutron channels
    neutron_region = (gauss_x >= gamma_border) & (gauss_x <= neutron_border)
    ax.fill_between(gauss_x, gauss_neutron, where=neutron_region, color=bg_red)
    ax.fill_between(gauss_x, gauss_neutron, where=~neutron_region, color=bg_bluegrey)
    ax.fill_between(gauss_x, gauss_gamma, color=bg_bluegrey)

    ax.set_xlabel("PSD")
    ax.set_ylabel("Counts (x1000)")
    
    ax.set_xlim(0, 0.55)
    ax.set_ylim(0, 3300)
    
    ax.yaxis.set_major_formatter(lambda x, _: f"{x / 1000:.1f}")
    ax.xaxis.set_major_formatter(lambda x, _: "" if x == 0 else f"{x:.1f}")
    ax.tick_params(labelsize=fontsize_small)

In [ ]:
def plot_figure_4c(ax: mpl.axes.Axes):
    cmap = plt.colormaps["viridis"]
    angle_elev = 30
    angle_rot = -20
    e_margin = 2
    psd_margin = 0.002

    df = get_figure_4c_data(e_margin, psd_margin)
    x = df["x"]
    y = df["y"]
    z = df["z"]
    dx = df["dx"]
    dy = df["dy"]
    dz = df["dz"]
    within_borders = df["within_borders"]

    # Neutron window colormap
    min_dz = -2000
    max_dz = np.max(dz)
    norm = mpl.colors.Normalize(vmin=min_dz, vmax=max_dz)
    mapped_colors = [cmap(norm(dz_val)) for dz_val in dz]
    mapped_colors = [
        bg_red if is_within_borders else color
        for is_within_borders, color in zip(within_borders, mapped_colors)
    ]

    # Neutron window borders
    border_e = np.linspace(lower_energy_bound, 3000, 200, dtype="float")
    bottom_border_psds = borders.bottom(border_e)
    top_border_psds = borders.top(border_e)
    border_zs = np.full_like(border_e, 0.000001, dtype="float")

    ax.view_init(angle_elev, angle_rot)
    ax.bar3d(
        x, y, z, dx, dy, dz,
        color=mapped_colors,
        shade=False,
        zsort="max",
        lw=0.05,
        ec="black"
    )

    # Neutron window
    ax.plot(border_e, bottom_border_psds, zs=0, zdir='z', color="red", zorder=0)
    ax.plot(border_e, top_border_psds, zs=0, zdir='z', color="red", zorder=0)
    ax.fill_between(
        border_e, bottom_border_psds, border_zs,
        border_e, top_border_psds, border_zs,
        color="red", alpha=0.5, zorder=0
    )

    ax.set_xlabel("Energy (ADC channel x1000)")
    ax.set_ylabel("PSD")
    ax.set_zlabel("Counts (x1000)")
    
    ax.set_xlim(0, 3000)
    ax.set_ylim(0, 0.5)
    ax.set_zlim(0, 4000)

    set_3d_axis_label_padding(ax, -1.3, -1.2, -1.5)
    
    ax.xaxis.set_major_formatter(lambda x, pos: f"{x / 1000:.1f}")
    ax.zaxis.set_major_formatter(lambda z, pos: f"{z / 1000:.1f}")
    ax.tick_params(labelsize=fontsize_small)
    set_3d_ticklabel_padding(ax, -0.8, -0.7, -1.0)
    set_3d_ticklabel_alignment(ax)
    
    set_3d_panes_transparent(ax)

In [ ]:
def plot_figure_4d(ax: mpl.axes.Axes):
    df = get_figure_4d_main_data()
    time_series = df.index
    rate_series = df["rate"]

    ax.errorbar(
        time_series,
        rate_series,
        linestyle=':',
        markersize=0,
        color="black"
    )
    ax.bar(
        time_series,
        rate_series,
        bin_length / 60,
        edgecolor="black",
        facecolor="#00000000"
    )
    
    ax.set_xlabel("Time [minutes]")
    ax.set_ylabel("Neutron count rate [1/s]")
    
    ax.set_xlim(0, 130)
    ax.set_ylim(0, 300)

    ax.tick_params(labelsize=fontsize_small)

In [ ]:
def plot_figure_4d_side(ax: mpl.axes.Axes, bin_idx: int):
    cmap = plt.colormaps["viridis"]
    angle_elev = 30
    angle_rot = -20
    e_margin = 2
    psd_margin = 0.002

    df = get_figure_4d_side_data(bin_idx, e_margin, psd_margin)
    x = df["x"]
    y = df["y"]
    z = df["z"]
    dx = df["dx"]
    dy = df["dy"]
    dz = df["dz"]
    within_borders = df["within_borders"]

    # Neutron window colormap
    min_dz = 0
    max_dz = np.max(dz)
    norm = mpl.colors.Normalize(vmin=min_dz, vmax=max_dz)
    mapped_colors = [cmap(norm(dz_val)) for dz_val in dz]
    mapped_colors = [
        bg_red if is_within_borders else color
        for is_within_borders, color in zip(within_borders, mapped_colors)
    ]

    # Neutron window borders
    border_e = np.linspace(lower_energy_bound, 3000, 200, dtype="float")
    bottom_border_psds = borders.bottom(border_e)
    top_border_psds = borders.top(border_e)
    border_zs = np.zeros_like(border_e, dtype="float")

    ax.view_init(angle_elev, angle_rot)
    ax.bar3d(x, y, z, dx, dy, dz,
             color=mapped_colors,
             shade=False,
             zsort="max",
             lw=0.05,
             ec="black"
            )

    # Neutron window
    ax.plot(
        border_e,
        bottom_border_psds,
        zs=0,
        zdir='z',
        axlim_clip=True,
        color="red",
        zorder=0
    )
    ax.plot(
        border_e,
        top_border_psds,
        zs=0,
        zdir='z',
        axlim_clip=True,
        color="red",
        zorder=0
    )
    ax.fill_between(
        border_e, bottom_border_psds, border_zs,
        border_e, top_border_psds, border_zs,
        axlim_clip=True, color="red", alpha=0.5, zorder=0
    )

    ax.set_xlim(0, 3000)
    ax.set_ylim(0, 0.5)
    ax.set_zlim(0, 200)
    
    ax.tick_params(
        labelleft=False, labelright=False, labelbottom=False, labeltop=False,
    )

    set_3d_panes_transparent(ax)

### Plot Creation

In [ ]:
fig_folder = dl_folder / "figures"
fig_folder.mkdir(parents=True, exist_ok=True)

In [ ]:
cm = 1/2.54
mm = 0.1*cm

# fig_height = 22
fig_height_mm = 205
fig_height = fig_height_mm*mm
fig_width_mm = 180
fig_width = fig_width_mm*mm
print(f"Figure dimensions: {fig_width_mm} mm x {fig_height_mm} mm ({fig_width:.2f} in. x {fig_height:.2f} in.)")

row_ratios = [2, 1, 1, 2]
row_total = sum(row_ratios)
row_heights = [(ratio / row_total) * fig_height for ratio in row_ratios]
row_1, row_2, row_3, row_4 = row_heights
height_a = row_1
height_b = row_1
height_c = row_2 + row_3
height_d_side1 = row_2
height_d_side2 = row_2
height_d_main = row_3
height_e = row_4
heights = [
    height_a, height_b, height_c,
    height_d_side1, height_d_side2, height_d_main,
    height_e
]

# fig_width = 17
col_ratios = [2, 1, 1]
col_total = sum(col_ratios)
col_widths = [(ratio / col_total) * fig_width for ratio in col_ratios]
col_1, col_2, col_3 = col_widths
width_a = col_1
width_b = col_2 + col_3
width_c = col_1
width_d_side1 = col_2
width_d_side2 = col_3
width_d_main = col_2 + col_3
width_e = sum(col_widths)
widths = [
    width_a, width_b, width_c,
    width_d_side1, width_d_side2, width_d_main,
    width_e
]

fig_sizes = list(zip(widths, heights))
figsize_a, figsize_b, figsize_c, *rest = fig_sizes
figsize_d_side1, figsize_d_side2, figsize_d_main, *rest = rest
figsize_e, *_ = rest

fig_names = [f"Figure {x}" for x in ["A", "B", "C", "D side 1", "D side 2", "D main", "E"]]
for fig_name, figsize in zip(fig_names, fig_sizes):
    width, height = figsize
    print(f"{fig_name}: {width:.2f} in. wide x {height:.2f} in. tall")

In [ ]:
def save_figure(
    fig: mpl.figure.Figure,
    filename: str,
    extensions: list[str] | None = None,
    **kwargs
):
    if extensions is None:
        extensions = ["png", "eps", "pdf"]
    paths = [fig_folder / f"{filename}.{ext}" for ext in extensions]
    for path in paths:
        fig.savefig(path, **kwargs)

In [ ]:
# Fig. 4a
fig, ax = plt.subplots(
    1, 1,
    figsize=figsize_a,
    dpi=600,
    subplot_kw={"projection": "3d"},
)
plot_figure_4a(ax)
ax.set_box_aspect(None, zoom=0.85)
save_figure(fig, "fig_4a", dpi=fig.dpi)

In [ ]:
# Fig. 4b
fig, ax = plt.subplots(
    1, 1,
    figsize=figsize_b,
    dpi=600,
    layout="constrained",
)
plot_figure_4b(ax)
save_figure(fig, "fig_4b", dpi=fig.dpi, bbox_inches="tight")

In [ ]:
# Fig. 4c
fig, ax = plt.subplots(
    1, 1,
    figsize=figsize_c,
    dpi=600,
    subplot_kw={"projection": "3d"},
)
plot_figure_4c(ax)
ax.set_box_aspect(None, zoom=0.85)
save_figure(fig, "fig_4c", dpi=fig.dpi)

In [ ]:
# Fig. 4d_main
fig, ax = plt.subplots(
    1, 1,
    figsize=figsize_d_main,
    dpi=600,
    layout="constrained",
)
plot_figure_4d(ax)
save_figure(fig, "fig_4d_main", dpi=fig.dpi, bbox_inches="tight")

In [ ]:
# Fig. 4d_upper_left
fig, ax = plt.subplots(
    1, 1,
    figsize=figsize_d_side1,
    dpi=600,
    subplot_kw={"projection": "3d"},
    layout="constrained",
)
plot_figure_4d_side(ax, 0)
save_figure(fig, "fig_4d_side1", dpi=fig.dpi)

In [ ]:
# Fig. 4d_upper_right
fig, ax = plt.subplots(
    1, 1,
    figsize=(width_d_side2, height_d_side2),
    dpi=600,
    subplot_kw={"projection": "3d"},
    layout="constrained",
)
plot_figure_4d_side(ax, 2)
save_figure(fig, "fig_4d_side2", dpi=fig.dpi)

In [ ]:
input("Processing done, hit Enter to finish")
stop()

## Base Data Export

In [ ]:
excel_folder = dl_folder / "excel"
excel_folder.mkdir(parents=True, exist_ok=True)

In [ ]:
def export_figure_4a_data():
    df = get_figure_4a_data()
    df.to_excel(
        excel_folder / "figure_4a.xlsx",
    )


def export_figure_4b_data():
    slice_df, gaussian_df, fit_df = get_figure_4b_data()
    slice_df.to_excel(
        excel_folder / "figure_4b_slice.xlsx",
        header=["Counts"],
        index_label="PSD"
    )
    gaussian_df.to_excel(
        excel_folder / "figure_4b_gaussian.xlsx",
        header=["Neutron", "Gamma"],
        index_label="PSD"
    )
    fit_df.to_excel(
        excel_folder / "figure_4b_vline_positions.xlsx",
        header=["Neutron", "Gamma"],
        index_label="Line Type"
    )


def export_figure_4c_data():
    df = get_figure_4c_data()
    df.to_excel(
        excel_folder / "figure_4c.xlsx",
        header=["x", "y", "z", "dx", "dy", "dz", "Within Borders"]
    )


def export_figure_4d_main_data():
    df = get_figure_4d_main_data()
    df.to_excel(
        excel_folder / "figure_4d_main.xlsx",
        header=["Rate"],
        index_label="Time (min)"
    )


def export_figure_4d_side_data():
    left_df = get_figure_4d_side_data(0)
    right_df = get_figure_4d_side_data(2)
    left_df.to_excel(
        excel_folder / "figure_4d_side_left.xlsx",
        header=["x", "y", "z", "dx", "dy", "dz", "Within Borders"]
    )
    right_df.to_excel(
        excel_folder / "figure_4d_side_right.xlsx",
        header=["x", "y", "z", "dx", "dy", "dz", "Within Borders"]
    )

In [ ]:
export_figure_4a_data()

In [ ]:
export_figure_4b_data()

In [ ]:
export_figure_4c_data()

In [ ]:
export_figure_4d_main_data()

In [ ]:
export_figure_4d_side_data()

In [ ]:
input("Processing done, hit Enter to finish")
stop()